# OODKit showcase: ImageNet val (ID) vs ImageNet-O (OOD)

Run cell-by-cell in **VS Code / Cursor** (“Run Current Cell”) or **Jupyter** (`# %%` markers).

**Requirements:** `pip install -e ".[ml]"` plus HuggingFace access for the backbone if needed.

**Data layout**
- **ID (val):** one folder per WordNet id (`n01498041/…`), same as `SynsetImageDataset`.
- **OOD (e.g. ImageNet-O):** same folder layout (synset subfolders + images).
- **`LOC_synset_mapping.txt`:** ILSVRC 1000-class list (line order = canonical index).

Edit the **paths** and **training** constants in the first code cell below.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# --- Edit these paths for your machine ---------------------------------------
DATASETS_ROOT = Path("/datasets")
IMAGENET_VAL_ROOT = DATASETS_ROOT / "imagenet-val"
IMAGENET_O_ROOT = DATASETS_ROOT / "imagenet-o"
LOC_SYNSET_MAPPING = DATASETS_ROOT / "LOC_synset_mapping.txt"

BACKBONE = "dinov2-small"
HEAD_EPOCHS = 10
BATCH_SIZE = 64
NUM_WORKERS = 4
# DataLoader: use pinned host memory and keep workers alive between epochs (GPU training).
PIN_MEMORY = True
PERSISTENT_WORKERS = True

SEED = 42
# Fraction of **full val** used to train the linear head and fit detectors; rest is ID test in the mix.
TRAIN_FRACTION = 0.9

_DL_KW = dict(
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)


def _check_paths() -> None:
    missing = [p for p in (IMAGENET_VAL_ROOT, IMAGENET_O_ROOT, LOC_SYNSET_MAPPING) if not p.exists()]
    if missing:
        print("Missing paths — update the constants at the top of this cell:", file=sys.stderr)
        for p in missing:
            print(f"  - {p}", file=sys.stderr)
        raise FileNotFoundError("Fix IMAGENET_VAL_ROOT, IMAGENET_O_ROOT, and LOC_SYNSET_MAPPING.")


_check_paths()

In [2]:
import numpy as np
import torch
from torch.utils.data import Dataset

from oodkit.contrib.imagenet import SynsetTable, SynsetImageDataset
from oodkit.data.features import Features
from oodkit.detectors import Energy, KNN, MSP, Mahalanobis, PCA, PCAFusion, ViM, WDiscOOD
from oodkit.embeddings.backbones import load_backbone
from oodkit.embeddings.embedder import Embedder
from oodkit.evaluation import ScoreBank, concatenate_embedding_results, evaluate

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class IndexedDatasetSubset(Dataset):
    """Subset by indices while keeping ``imgs`` / ``targets`` / ``classes`` for ``Embedder``."""

    def __init__(self, base: SynsetImageDataset, indices: list[int]) -> None:
        self.base = base
        self.indices = list(indices)
        self.classes = base.classes
        self.wnids = base.wnids
        self.imgs = [base.imgs[i] for i in self.indices]
        self.targets = [lbl for _, lbl in self.imgs]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        return self.base[self.indices[i]]

In [4]:
synset_table = SynsetTable.from_file(LOC_SYNSET_MAPPING)
_, processor, _ = load_backbone(BACKBONE)

val_full = SynsetImageDataset(IMAGENET_VAL_ROOT, synset_table, processor, strict=True)
ood_full = SynsetImageDataset(IMAGENET_O_ROOT, synset_table, processor, strict=False)
print("synset image datasets made")

rng = np.random.default_rng(SEED)
n_val = len(val_full)
if n_val < 2:
    raise RuntimeError(f"Need at least 2 val images; got {n_val} under {IMAGENET_VAL_ROOT}")

n_train = max(1, min(int(TRAIN_FRACTION * n_val), n_val - 1))
n_id_test = n_val - n_train

perm = rng.permutation(n_val)
train_idx = perm[:n_train].tolist()
id_test_idx = perm[n_train : n_train + n_id_test].tolist()
print("Permutations done")

train_ds = IndexedDatasetSubset(val_full, train_idx)
id_test_ds = IndexedDatasetSubset(val_full, id_test_idx)
ood_ds = ood_full

print(f"Val: train={len(train_ds)}  id_test={len(id_test_ds)}  |  ImageNet-O (full): {len(ood_ds)}")

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 3690.44it/s]


synset image datasets made
Permutations done
Val: train=45000  id_test=5000  |  ImageNet-O (full): 2000


In [5]:
# Train classifier head on ID train split (backbone frozen in mode="head")
embedder = Embedder(backbone=BACKBONE)
embedder.fit(
    train_ds,
    mode="head",
    epochs=HEAD_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=1e-3,
    save=False,
    **_DL_KW,
)

training (head): 100%|██████████| 10/10 [06:32<00:00, 39.22s/it, avg_loss=0.0585]


In [6]:
# Extract embeddings + logits once per split (full ID test + full ImageNet-O)
train_res = embedder.extract(train_ds, batch_size=BATCH_SIZE, **_DL_KW)
id_test_res = embedder.extract(id_test_ds, batch_size=BATCH_SIZE, **_DL_KW)
ood_res = embedder.extract(ood_ds, batch_size=BATCH_SIZE, **_DL_KW)

assert train_res.logits is not None and train_res.labels is not None

combined, ood_labels = concatenate_embedding_results([id_test_res, ood_res], [0, 1])
comb_feat = combined.to_features()

id_train_feat = train_res.to_features()
y_train = train_res.labels
y_combined_class = combined.labels

# Free per-split results (data lives on in combined / id_train_feat)
del id_test_res, ood_res

# Release GPU memory back to system after extraction
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Combined:", combined.embeddings.shape, "ood_labels:", ood_labels.shape)

Combined: (7000, 384) ood_labels: (7000,)


In [7]:
# Build detectors: fit on ID **train** only, score on ID test + OOD
head = embedder._head  # noqa: SLF001 — showcase script; no public accessor yet
assert head is not None
W = head.weight.detach().cpu().numpy()
b = head.bias.detach().cpu().numpy()

In [8]:
detectors: dict = {}
print("MSP")
msp = MSP()
msp.fit(id_train_feat)
detectors["MSP"] = msp

print("energy")
energy = Energy()
energy.fit(id_train_feat)
detectors["Energy"] = energy

print("mahala")
maha = Mahalanobis()
maha.fit(id_train_feat, y=y_train)
detectors["Mahalanobis"] = maha

print("knn")
knn = KNN(k=min(10, len(train_ds)), backend="auto")
knn.fit(id_train_feat)
detectors["KNN"] = knn

print("PCA")
pca = PCA(kernel="linear")
pca.fit(id_train_feat)
detectors["PCA"] = pca

print("PCAFusion")
pcaf = PCAFusion(kernel="linear")
pcaf.fit(id_train_feat)
detectors["PCAFusion"] = pcaf

print("VIM")
vim = ViM(W, b)
vim.fit(id_train_feat)
detectors["ViM"] = vim

print("Cosine PCA")
cpca = PCA(kernel="cosine")
cpca.fit(id_train_feat)
detectors["cosinePCA"] = cpca

print("RFF Cosine PCA")
rpca = PCA(kernel="rff_cosine")
rpca.fit(id_train_feat)
detectors["rffcosinePCA"] = rpca

print("WDiscOOD")
wd = WDiscOOD()
wd.fit(id_train_feat, y=y_train)
detectors["WDiscOOD"] = wd

# Free training features/labels — detectors keep only what they need internally
del id_train_feat, y_train, train_res, W, b

print("Done fitting")

MSP
energy
mahala
knn
PCA
PCAFusion
VIM
Cosine PCA
RFF Cosine PCA
WDiscOOD


In [9]:
bank = ScoreBank(ood_labels=ood_labels, class_labels=y_combined_class)
for name, det in detectors.items():
    bank.add(name, det.score(comb_feat))

table = evaluate(bank)
print(table)

Detector            AUROC     FPR@95  AUPR(OOD)   AUPR(ID)     DetErr
---------------------------------------------------------------------
MSP                0.7314     0.7086     0.5021     0.8736     0.3266
Energy             0.7761     0.6304     0.5114     0.8975     0.2846
Mahalanobis        0.8575     0.4706     0.6769     0.9396     0.2208
KNN                0.8325     0.5626     0.6453     0.9248     0.2489
PCA                0.7015     0.8128     0.4732     0.8453     0.3507
PCAFusion          0.7922     0.6152     0.5514     0.9064     0.2720
ViM                0.7921     0.6170     0.5393     0.9079     0.2719
cosinePCA          0.7059     0.7932     0.4731     0.8507     0.3464
rffcosinePCA       0.5820     0.9102     0.3474     0.7668     0.4364
WDiscOOD           0.8562     0.5074     0.6998     0.9372     0.2193


Higher scores ⇒ more OOD. **AUROC** / **FPR@95** / **AUPR** use `ood_labels`: ID test = 0, ImageNet-O = 1.

In [10]:
# from oodkit.evaluation import evaluate_by_class
# by_cls = evaluate_by_class(bank)